# LLM Generate

Run text generation on a list of prompts using transformers, vLLM, or API backends.

In [ ]:
#|default_exp llm

In [ ]:
#|export
from llm_runner.models import load_llm

In [ ]:
#|export
def run_llm_generate(prompts: list[str], *, model_name: str = "Qwen/Qwen2.5-7B-Instruct",
                     device: str = "cuda", dtype: str = "float16",
                     backend: str = "transformers", max_new_tokens: int = 60,
                     batch_size: int = 128,
                     system_message: str | None = None,
                     json_schema: dict | None = None,
                     tensor_parallel_size: int = 1,
                     vllm_kwargs: dict | None = None,
                     temperature: float = 0.0,
                     top_p: float = 1.0,
                     top_k: int = -1) -> list[str]:
    """Generate text completions for a list of prompts.

    Args:
        prompts: List of prompt strings.
        model_name: HuggingFace model ID (or API model name for api backend).
        device: Device ("cuda", "cpu"). Ignored for api backend.
        dtype: Model precision. Ignored for api backend.
        backend: "transformers", "vllm", or "api".
        max_new_tokens: Maximum tokens to generate per prompt.
        batch_size: Number of prompts per batch (transformers backend only).
        system_message: Optional system message prepended to each prompt.
        json_schema: Optional JSON schema dict to constrain output to valid JSON.
        tensor_parallel_size: Number of GPUs for tensor parallelism (vllm only, default 1).
        vllm_kwargs: Extra kwargs passed to vLLM's LLM constructor (vllm only).
        temperature: Sampling temperature (0.0 = greedy).
        top_p: Nucleus sampling threshold (1.0 = disabled).
        top_k: Top-k sampling (-1 = disabled).

    Returns:
        List of generated response strings, one per prompt.
    """
    llm = load_llm(model_name, device=device, dtype=dtype, backend=backend,
                   tensor_parallel_size=tensor_parallel_size,
                   vllm_kwargs=vllm_kwargs)

    sampling_kwargs = dict(temperature=temperature, top_p=top_p, top_k=top_k)

    # For api and vllm backends, send all prompts at once (they handle batching internally)
    if backend in ("api", "vllm"):
        return llm.generate(
            prompts,
            max_new_tokens=max_new_tokens,
            system_message=system_message,
            json_schema=json_schema,
            **sampling_kwargs,
        )

    # For transformers backend, batch manually
    all_responses = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        responses = llm.generate(
            batch,
            max_new_tokens=max_new_tokens,
            system_message=system_message,
            json_schema=json_schema,
            **sampling_kwargs,
        )
        all_responses.extend(responses)

    return all_responses